In [10]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader

### load all txt files in the directory
dir_loader = DirectoryLoader(
    "../data/dukcapil_pdf", 
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False)

pdf_docs = dir_loader.load()
pdf_docs

[Document(metadata={'producer': '', 'creator': 'Canon', 'creationdate': '2023-10-31T08:44:01+07:00', 'source': '..\\data\\dukcapil_pdf\\Buku-Saku-Dafduk-Capil-2023.pdf', 'file_path': '..\\data\\dukcapil_pdf\\Buku-Saku-Dafduk-Capil-2023.pdf', 'total_pages': 282, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-10-31T09:11:04+07:00', 'trapped': '', 'modDate': "D:20231031091104+07'00'", 'creationDate': "D:20231031084401+07'00'", 'page': 0}, page_content='a\na\ne\na\na\no\n:Hr@ Pyffi\nBeTAKHLAX\na\na\na\na\na\n7\na\na\na\na\n-TAHUN aoa-:'),
 Document(metadata={'producer': '', 'creator': 'Canon', 'creationdate': '2023-10-31T08:44:01+07:00', 'source': '..\\data\\dukcapil_pdf\\Buku-Saku-Dafduk-Capil-2023.pdf', 'file_path': '..\\data\\dukcapil_pdf\\Buku-Saku-Dafduk-Capil-2023.pdf', 'total_pages': 282, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-10-31T09:11:04+07:00', 'trapped': '', 'modDate': "D:

In [11]:
print(f"Total pages extracted: {len(pdf_docs)}")
print(f"Page 0: {pdf_docs[0].page_content[:100]}")
print(f"Page 1: {pdf_docs[1].page_content[:100]}")
print(f"Page 5: {pdf_docs[5].page_content[:100]}")

Total pages extracted: 282
Page 0: a
a
e
a
a
o
:Hr@ Pyffi
BeTAKHLAX
a
a
a
a
a
7
a
a
a
a
-TAHUN aoa-:
Page 1: KATA PENGANTAR
Dengan penuh rasa syukur kepada Tuhan Yang
Maha Esa, kami ingin menyampaikan terima k
Page 5: 4. Apakah penerbitan KTP-el dapat dilakukan di
luar kabupaten/kota alamat domisili 
yang
tertera dal


In [12]:
import re

for doc in pdf_docs:
    page_num = doc.metadata['page']
    text = doc.page_content
    
    # hitung jumlah karakter total
    total_chars = len(text)
    
    # hitung karakter yang "normal" (huruf, angka, spasi)
    normal_chars = len(re.findall(r'[a-zA-Z0-9\s]', text))
    
    # hitung rasio karakter normal
    ratio = normal_chars / total_chars if total_chars > 0 else 0
    
    print(f"Halaman {page_num} | Total chars: {total_chars} | Ratio normal: {ratio:.2f}")

Halaman 0 | Total chars: 65 | Ratio normal: 0.92
Halaman 1 | Total chars: 1152 | Ratio normal: 0.98
Halaman 2 | Total chars: 507 | Ratio normal: 0.97
Halaman 3 | Total chars: 1186 | Ratio normal: 0.49
Halaman 4 | Total chars: 1104 | Ratio normal: 0.73
Halaman 5 | Total chars: 1027 | Ratio normal: 0.75
Halaman 6 | Total chars: 1127 | Ratio normal: 0.73
Halaman 7 | Total chars: 1110 | Ratio normal: 0.69
Halaman 8 | Total chars: 1009 | Ratio normal: 0.70
Halaman 9 | Total chars: 1187 | Ratio normal: 0.71
Halaman 10 | Total chars: 1203 | Ratio normal: 0.76
Halaman 11 | Total chars: 1139 | Ratio normal: 0.67
Halaman 12 | Total chars: 1105 | Ratio normal: 0.76
Halaman 13 | Total chars: 1165 | Ratio normal: 0.76
Halaman 14 | Total chars: 1167 | Ratio normal: 0.78
Halaman 15 | Total chars: 1128 | Ratio normal: 0.82
Halaman 16 | Total chars: 987 | Ratio normal: 0.76
Halaman 17 | Total chars: 1070 | Ratio normal: 0.77
Halaman 18 | Total chars: 1090 | Ratio normal: 0.76
Halaman 19 | Total chars: 

## Step 1 — Filter Halaman

Buang halaman yang tidak berguna untuk RAG:
- **Halaman 0 & 281**: cover scan (sampah)
- **Halaman 3–24**: daftar isi (penuh titik-titik & nomor halaman, tidak ada nilai konten)

Opsional dibuang: halaman 1–2 (kata pengantar) — tergantung apakah relevan untuk pertanyaan user.

In [13]:
COVER_PAGES = {0, 281}          # scan sampah
DAFTAR_ISI_PAGES = set(range(3, 25))  # page 3–24 (0-indexed), ratio rendah

SKIP_PAGES = COVER_PAGES | DAFTAR_ISI_PAGES

filtered_docs = [doc for doc in pdf_docs if doc.metadata['page'] not in SKIP_PAGES]

print(f"Sebelum filter: {len(pdf_docs)} halaman")
print(f"Sesudah filter : {len(filtered_docs)} halaman")
print(f"Dibuang        : {sorted(SKIP_PAGES)}")

Sebelum filter: 282 halaman
Sesudah filter : 258 halaman
Dibuang        : [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 281]


## Step 2 — Clean Teks per Halaman

Masalah yang perlu diatasi:
1. Sisa titik-titik daftar isi yang bisa bocor ke halaman awal (`......`)
2. Nomor halaman standalone di akhir baris (mis. `\n18\n`)
3. Whitespace berlebihan (tab, spasi ganda, baris kosong berulang)
4. Newline di tengah kalimat karena layout dua kolom / word-wrap PDF

In [14]:
import re
import copy

def clean_page(text: str) -> str:
    # 1. Hapus deretan titik (artefak daftar isi)
    text = re.sub(r'\.{3,}', '', text)
    
    # 2. Hapus nomor halaman standalone (baris hanya berisi angka)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    
    # 3. Gabungkan baris yang terpotong di tengah kalimat:
    #    baris yang berakhir bukan dengan tanda baca → sambung dengan spasi
    text = re.sub(r'(?<![.!?:\-])\n(?=[a-zA-Z])', ' ', text)
    
    # 4. Normalisasi whitespace
    text = re.sub(r'[ \t]+', ' ', text)          # spasi ganda → satu
    text = re.sub(r'\n{3,}', '\n\n', text)       # baris kosong berulang → maks 2
    
    return text.strip()


cleaned_docs = []
for doc in filtered_docs:
    new_doc = copy.copy(doc)
    new_doc.page_content = clean_page(doc.page_content)
    if new_doc.page_content:   # buang kalau setelah cleaning jadi kosong
        cleaned_docs.append(new_doc)

print(f"Dokumen setelah cleaning: {len(cleaned_docs)} halaman")

# Spot-check
for page_idx in [1, 25, 26, 100, 278]:
    candidates = [d for d in cleaned_docs if d.metadata['page'] == page_idx]
    if candidates:
        print(f"\n=== HALAMAN {page_idx} (setelah cleaning) ===")
        print(candidates[0].page_content[:400])

Dokumen setelah cleaning: 258 halaman

=== HALAMAN 1 (setelah cleaning) ===
KATA PENGANTAR Dengan penuh rasa syukur kepada Tuhan Yang Maha Esa, kami ingin menyampaikan terima kasih atas anugerah rahmat dan petunjuk-Nya. Buku Saku Pendaftaran Penduduk dan Pencatatan Sipil untuk Dinas Kependudukan dan Pencatatan Sipil Provinsi, Kabupaten/Kota telah berhasil diselesaikan. Buku ini disajikan dengan harapan dapat memberikan panduan yang bermanfaat bagi aparat penyelenggara lay

=== HALAMAN 25 (setelah cleaning) ===
BAB I PENDAHULUAN A.LATAR BELAKANG Administrasi kependudukan di Indonesia merupakan hal yang sangat berperan dalam pembangunan, dimana dari sistem administrasi penduduk tersebut dapat diketahui tentang data-data penduduk dan informasi yang sesuai dengan keadaan penduduk dan tentang kondisi daerah tempat tinggal penduduk.
Negara Kesatuan Republik Indonesia (NKRI) pada hakikatnya berkewajiban member

=== HALAMAN 26 (setelah cleaning) ===
ntah blik implikasi/pengaruh terhadap peruba

## Step 3 — Verifikasi Kualitas Final

In [15]:
import statistics

lengths = [len(d.page_content) for d in cleaned_docs]
ratios = [len(re.findall(r'[a-zA-Z0-9\s]', d.page_content)) / len(d.page_content)
          for d in cleaned_docs if len(d.page_content) > 0]

print(f"Total halaman  : {len(cleaned_docs)}")
print(f"Chars rata-rata: {statistics.mean(lengths):.0f}")
print(f"Chars min/max  : {min(lengths)} / {max(lengths)}")
print(f"Ratio bersih   : min={min(ratios):.2f}  mean={statistics.mean(ratios):.2f}  max={max(ratios):.2f}")

# Flagging halaman yang masih mencurigakan (ratio < 0.85)
suspicious = [(d.metadata['page'], len(d.page_content),
               len(re.findall(r'[a-zA-Z0-9\s]', d.page_content)) / len(d.page_content))
              for d in cleaned_docs if len(d.page_content) > 0
              and len(re.findall(r'[a-zA-Z0-9\s]', d.page_content)) / len(d.page_content) < 0.85]

if suspicious:
    print(f"\nHalaman masih mencurigakan (ratio < 0.85):")
    for page, chars, ratio in suspicious:
        print(f"  Halaman {page:3d} | chars={chars:4d} | ratio={ratio:.2f}")
else:
    print("\nSemua halaman bersih (ratio >= 0.85)")

Total halaman  : 258
Chars rata-rata: 922
Chars min/max  : 144 / 1183
Ratio bersih   : min=0.95  mean=0.97  max=0.99

Semua halaman bersih (ratio >= 0.85)


## Step 4 — Tag Section Metadata

Tag setiap halaman dengan section asal (BAB I/II/III/Kata Pengantar). Metadata ini akan dipakai chunker di tahap berikutnya untuk strategi chunking yang berbeda per section (Q&A-aware untuk BAB II, character-based untuk BAB I/III).

In [16]:
def tag_section(page: int) -> str:
    if page in (1, 2):
        return "Kata Pengantar"
    if 25 <= page <= 31:
        return "BAB I - Pendahuluan"
    if 32 <= page <= 277:
        return "BAB II - Pertanyaan dan Jawaban"
    if 278 <= page <= 280:
        return "BAB III - Penutup"
    return "Unknown"

for doc in cleaned_docs:
    doc.metadata["section"] = tag_section(doc.metadata["page"])

# Distribusi section
from collections import Counter
section_counts = Counter(d.metadata["section"] for d in cleaned_docs)
for section, count in section_counts.items():
    print(f"{section:40s} : {count:3d} halaman")

Kata Pengantar                           :   2 halaman
BAB I - Pendahuluan                      :   7 halaman
BAB II - Pertanyaan dan Jawaban          : 246 halaman
BAB III - Penutup                        :   3 halaman


## Step 5 — Export ke Pickle

Dump `cleaned_docs` ke pickle supaya bisa dipakai langsung di `build_vectorstore.ipynb` tanpa perlu rerun preprocessing dari awal.

In [ ]:
import pickle
from pathlib import Path

OUTPUT_PATH = Path("../data/cleaned_docs.pkl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(cleaned_docs, f)

print(f"Saved {len(cleaned_docs)} docs to {OUTPUT_PATH.resolve()}")

Saved 258 docs to C:\Users\Nafisha\Documents\RAGTrial\data\cleaned_docs.pkl


: 

## Step 6 — Export ke JSON (untuk inspeksi manual)

Dump versi JSON yang isinya **identik** dengan pickle (list of `{page_content, metadata}`). Gunanya untuk debugging/inspect tanpa harus unpickle di Python.

In [ ]:
import json

JSON_PATH = Path("../data/cleaned_docs.json")
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        [{"page_content": d.page_content, "metadata": d.metadata} for d in cleaned_docs],
        f, ensure_ascii=False, indent=2,
    )
print(f"Saved {len(cleaned_docs)} docs → {JSON_PATH.resolve()}")

# Verifikasi PKL == JSON (round-trip)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    json_docs = json.load(f)
assert len(json_docs) == len(cleaned_docs), "Count mismatch"
for i, (d, j) in enumerate(zip(cleaned_docs, json_docs)):
    assert d.page_content == j["page_content"], f"page_content mismatch at idx {i}"
    assert d.metadata == j["metadata"], f"metadata mismatch at idx {i}"
print("✓ PKL ↔ JSON content identical")